In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier # used for the DT Stump
from sklearn.ensemble import AdaBoostClassifier
from sklearn.pipeline import Pipeline

---
## <u>Generate Dataset</u>

In [2]:
X, y = make_classification(
    n_samples = 10000,
    n_features = 15,              
    n_informative = 12,
    n_redundant = 2,    # redundant features
    n_classes = 2,           
    random_state = 42
    
)

X = pd.DataFrame(X)

---
## <u>Train Test Split</u>

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
9254,3.223624,0.861537,2.381547,1.395780,4.014657,-1.527077,3.048176,1.264915,3.135539,3.286052,1.067258,-5.254813,-1.072045,-1.154300,-2.089999
1561,-3.577916,0.271841,-2.039949,4.737522,2.042567,2.968547,3.149826,4.125000,2.379183,0.855071,6.071050,7.298252,-6.449665,0.570726,-1.825491
1670,-3.274038,-1.223501,-2.915686,1.914367,-1.074814,0.016904,-1.189167,-1.681084,-0.641283,1.205698,-4.650563,-1.960073,4.951817,1.183464,-2.243373
6087,0.850901,-0.702375,-0.947497,2.046136,2.498039,1.364085,3.353895,-2.033510,2.892577,2.547747,2.607806,0.766182,-1.234793,0.483933,-1.407348
6669,0.471267,-0.092617,-2.245596,-1.257869,2.061824,-2.211428,0.891296,1.083817,-1.848658,0.857356,-1.442283,-0.577073,1.419910,0.391048,1.179483


---
## <u>Create and train model to make predictions</u>

In [4]:
# 1. We create a base model (DT Stump)

dt_stump = DecisionTreeClassifier(
    max_depth = 1,                       # forces the dt stump to make the simplest of predictions 
    random_state = 42                    # for repeatability
)

# 2. Ada_Boost_Classifier

ada_classifier = AdaBoostClassifier(
    estimator = dt_stump,                # if estimator = 'None', AdaBoostClassifier makes dt_stump on its own exactly like this
    n_estimators = 100,                  # No of weak learners
    random_state = 42
) 

ada_classifier.fit(X_train, y_train)

# 3. Predict

y_train_pred = ada_classifier.predict(X_train)
y_test_pred = ada_classifier.predict(X_test)

---
## <u>Evaluate</u>

In [5]:
print("For AdaBoost Classifier :-\n")

print("\nTraining scores :-")
print("Train Accuracy : ", accuracy_score(y_train, y_train_pred))
print("Train precision : ", precision_score(y_train, y_train_pred))
print("Train recall : ", recall_score(y_train, y_train_pred))
print("Train F1  : ", f1_score(y_train, y_train_pred))
print("Train confusion matrix : \n", confusion_matrix(y_train, y_train_pred))


print("\nTesting scores :-")
print("Test Accuracy : ", accuracy_score(y_test, y_test_pred))
print("Test precision : ", precision_score(y_test, y_test_pred))
print("Test recall : ", recall_score(y_test, y_test_pred))
print("Test F1 : ", f1_score(y_test, y_test_pred))
print("Test confusion matrix : \n", confusion_matrix(y_test, y_test_pred))

For AdaBoost Classifier :-


Training scores :-
Train Accuracy :  0.859375
Train precision :  0.8593596059113301
Train recall :  0.8629730398219144
Train F1  :  0.8611625323954091
Train confusion matrix : 
 [[3386  571]
 [ 554 3489]]

Testing scores :-
Test Accuracy :  0.8335
Test precision :  0.8158697863682605
Test recall :  0.8406708595387841
Test F1 :  0.8280846670108415
Test confusion matrix : 
 [[865 181]
 [152 802]]


---
## <u>Hyper Paramter Tuning</u>

In [8]:
# 1. make the pipeline

steps = [("ada", AdaBoostClassifier(
    random_state = 42,
    estimator = DecisionTreeClassifier(random_state = 42)
))]
pipeline = Pipeline(steps)

# 2. make parameter grid 

param_grid = {
    
    # paramters for the ada boost classifier
    "ada__n_estimators": [50, 100, 150],
    "ada__learning_rate": [0.1, 0.5, 1.0],

    # paramters for our estimator (DT stump) 
    "ada__estimator__max_depth": [1, 2, 3],
    "ada__estimator__min_samples_split": [5, 10, 20],
    "ada__estimator__ccp_alpha": [0.1, 0.01, 0.001]
}

# 3. cross validation

ada_classifier_cv = RandomizedSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    scoring = "accuracy",
    n_jobs = -1
    
)

# 4. train and predict

ada_classifier_cv.fit(X_train, y_train)
y_train_pred = ada_classifier_cv.predict(X_train)
y_test_pred = ada_classifier_cv.predict(X_test)

# 4. evaluate

print("For AdaBoost Classifier (hyperparameter tuning) :-\n")

print("Best Parameters : ", ada_classifier_cv.best_params_)

print("\nTraining scores :-")
print("Train Accuracy : ", accuracy_score(y_train, y_train_pred))
print("Train precision : ", precision_score(y_train, y_train_pred))
print("Train recall : ", recall_score(y_train, y_train_pred))
print("Train F1  : ", f1_score(y_train, y_train_pred))
print("Train confusion matrix : \n", confusion_matrix(y_train, y_train_pred))


print("\nTesting scores :-")
print("Test Accuracy : ", accuracy_score(y_test, y_test_pred))
print("Test precision : ", precision_score(y_test, y_test_pred))
print("Test recall : ", recall_score(y_test, y_test_pred))
print("Test F1 : ", f1_score(y_test, y_test_pred))
print("Test confusion matrix : \n", confusion_matrix(y_test, y_test_pred))

For AdaBoost Classifier (hyperparameter tuning) :-

Best Parameters :  {'ada__n_estimators': 150, 'ada__learning_rate': 0.1, 'ada__estimator__min_samples_split': 10, 'ada__estimator__max_depth': 3, 'ada__estimator__ccp_alpha': 0.001}

Training scores :-
Train Accuracy :  0.93125
Train precision :  0.9177230327672805
Train recall :  0.9490477368290873
Train F1  :  0.9331225680933852
Train confusion matrix : 
 [[3613  344]
 [ 206 3837]]

Testing scores :-
Test Accuracy :  0.9155
Test precision :  0.8976697061803445
Test recall :  0.9287211740041929
Test F1 :  0.9129314786192684
Test confusion matrix : 
 [[945 101]
 [ 68 886]]
